In [ ]:
repository_filter: list[str] = []
group_by: str = "repository"

In [ ]:
from moderne_visualizations_misc.reusable.data_loader import read_data_table
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import warnings

warnings.simplefilter("ignore")

# df = read_data_table("../samples/code_smells.csv")
df = read_data_table("../samples/v2/io.moderne.prethink.table.CodeSmells.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    group_col = "repoShort" if group_by == "repository" else "smellType"
    totals = df.groupby(group_col).size().sort_values(ascending=True)
    ordered_groups = totals.index.tolist()

    fig = go.Figure()
    for severity in qu.SEVERITY_ORDER:
        subset = df[df["severity"] == severity]
        counts = subset.groupby(group_col).size().reindex(ordered_groups, fill_value=0)
        fig.add_trace(
            go.Bar(
                y=counts.index,
                x=counts.values,
                name=severity,
                orientation="h",
                marker_color=qu.SEVERITY_COLORS[severity],
            )
        )
    fig.update_layout(
        barmode="stack",
        title="Code Smell Severity by "
        + ("Repository" if group_by == "repository" else "Smell Type"),
        xaxis_title="Count",
        yaxis_title="",
        height=max(400, len(ordered_groups) * 28 + 120),
        margin=dict(l=200, r=20, t=50, b=50),
        legend=dict(title="Severity"),
    )
fig.show()